# Chunking isolé du corpus candidat

Ce notebook découpe le dernier nettoyage candidat vers `data/experiments`. Le fichier réel `data/chunks/chunks.jsonl` est contrôlé par empreinte avant et après.


In [1]:
from pathlib import Path
from datetime import datetime
import hashlib, json, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
from chunker import HierarchicalChunker

cfg = load_config(ROOT / 'config.yaml')
experiment_root = ROOT / 'data' / 'experiments'
candidates = list(experiment_root.glob('cleaned_candidate_*'))
assert candidates, 'Exécutez d’abord le notebook 05.'
candidate_cleaned_dir = max(candidates, key=lambda path: path.stat().st_mtime)
candidate_chunks_path = experiment_root / f"chunks_candidate_{datetime.now():%Y%m%d_%H%M%S}.jsonl"
real_chunks_path = cfg.project.data_path / 'chunks' / 'chunks.jsonl'
real_hash_before = hashlib.sha256(real_chunks_path.read_bytes()).hexdigest()
print(f'Nettoyage candidat : {candidate_cleaned_dir}')
print(f'Chunks candidats : {candidate_chunks_path}')


Nettoyage candidat : <PROJECT_ROOT>\data\experiments\cleaned_candidate_20260803_121334
Chunks candidats : <PROJECT_ROOT>\data\experiments\chunks_candidate_20260803_121427.jsonl


In [2]:
chunker = HierarchicalChunker(cfg)
candidate_chunks = chunker.chunk_all(
    cleaned_dir=candidate_cleaned_dir,
    output_path=candidate_chunks_path,
)
print(f'{len(candidate_chunks)} chunks candidats construits.')


4416 chunks candidats construits.


In [3]:
with open(real_chunks_path, encoding='utf-8') as handle:
    current_chunks = [json.loads(line) for line in handle if line.strip()]
child_lengths = pd.Series([len(chunk.text) for chunk in candidate_chunks])
parent_lengths = pd.Series([len(chunk.parent_text) for chunk in candidate_chunks])
summary = pd.Series({
    'documents_candidats': len(list(candidate_cleaned_dir.glob('*.md'))),
    'chunks_actuels': len(current_chunks),
    'chunks_candidats': len(candidate_chunks),
    'delta_chunks': len(candidate_chunks) - len(current_chunks),
    'urls_actuelles': len({chunk['url'] for chunk in current_chunks}),
    'urls_candidates': len({chunk.url for chunk in candidate_chunks}),
    'enfant_p95_caractères': int(child_lengths.quantile(.95)),
    'enfant_max_caractères': int(child_lengths.max()),
    'parent_p95_caractères': int(parent_lengths.quantile(.95)),
    'parent_max_caractères': int(parent_lengths.max()),
})
display(summary.to_frame('valeur'))


,valeur
documents_candidats,932
chunks_actuels,4459
chunks_candidats,4416
delta_chunks,-43
urls_actuelles,1177
urls_candidates,930
enfant_p95_caractères,1595
enfant_max_caractères,1600
parent_p95_caractères,7195
parent_max_caractères,7200


In [4]:
targets = {
    'doctorants': 'https://ensea.ed.ci/formation-doctorale/doctorants/',
    'partenaires_recherche': 'https://ensea.ed.ci/cooperation/partenaires-de-recherche/',
}
target_rows = []
for name, url in targets.items():
    chunks = [chunk for chunk in candidate_chunks if chunk.url == url]
    target_rows.append({
        'page': name, 'chunks': len(chunks),
        'aperçu': ' '.join(chunk.text for chunk in chunks)[:1000],
    })
display(pd.DataFrame(target_rows))


,page,chunks,aperçu
0,doctorants,3,Doctorants\nAll\n2019\n2020\n2021\n2022\nCISEA...
1,partenaires_recherche,5,Partenaires de Recherche\nLe développement de ...


In [5]:
candidate_urls = [chunk.url.lower().split('?')[0] for chunk in candidate_chunks]
assert candidate_chunks_path.exists()
assert hashlib.sha256(real_chunks_path.read_bytes()).hexdigest() == real_hash_before
assert not any(url.endswith(('.jpg', '.jpeg', '.png', '.gif', '.webp')) for url in candidate_urls)
assert not any('/page/' in url and url.rstrip('/').split('/')[-1].isdigit() for url in candidate_urls)
assert any('Alain NDIKUMANA' in chunk.text for chunk in candidate_chunks)
assert any('INSD-Burkina Faso' in chunk.text for chunk in candidate_chunks)
print('Chunking candidat validé ; le fichier de chunks réel est resté strictement intact.')
print(f'Fichier candidat : {candidate_chunks_path}')


Chunking candidat validé ; le fichier de chunks réel est resté strictement intact.
Fichier candidat : <PROJECT_ROOT>\data\experiments\chunks_candidate_20260803_121427.jsonl
